# 3. Treinamento do Modelo

Este notebook orquestra o treinamento de modelos YOLO usando duas metodologias distintas para avaliar a redução da lacuna de domínio:
1. **Metodologia de Base:** Ajuste fino direto no domínio alvo (conjuntos de dados de interiores de ônibus).

2. **Metodologia Robusta (Em Etapas):** Pré-treinamento em um conjunto de dados geral de humanos (CrowdHuman) para desenvolver robustez à oclusão, seguido de ajuste fino no domínio alvo.

## Preparação e Configuracao global

In [ ]:
import sys
from pathlib import Path
from typing import Dict, Any, List, Optional

# Adiciona a raiz do projeto ao caminho para importação de módulos
sys.path.append("..")
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from ultralytics import YOLO
from src import config, datasets, train
from src import eval as eval_mod

# Defina a etapa de consumo de dados (deve ser 'processed' para treinamento)
DATASET_STAGE: str = "processed"

# Defina os agrupamentos de conjuntos de dados com base na metodologia
BUS_DATASETS: List[str] = [
    "inside-bus-view", 
    "passenger-deakin"
]

CROWD_DATASETS: List[str] = [
    "crowdhuman"
]

# Configuração básica de hiperparâmetros otimizada para 48 GB de VRAM (NVIDIA L40S) em comentário estará um exemplo de configuração para 6 GB de VRAM (NVIDIA RTX 3060) que pode ser usada para treinamento em máquinas com menos recursos.
TRAIN_CONFIG: Dict[str, Any] = {
    "epochs": 50,
    "patience": 15,
    "batch": 64,         # Maximiza a utilização da GPU
    "imgsz": 640,
    "workers": 8,        # Acelera o carregamento de dados
    "amp": True,
    "seed": 42,
    "deterministic": True,
    # Parâmetros de aumento de dados
    "degrees": 15.0,
    "translate": 0.1,
    "scale": 0.5,
    "mosaic": 1.0,
    "erasing": 0.4
}

## 3.1 Metodologia 1: Ajuste Fino Direto (Linha de Base)

Treina o modelo YOLO pré-treinado padrão diretamente nos conjuntos de dados de ônibus. Isso serve como nosso benchmark de controle.

In [1]:
def train_baseline() -> None:
    f"""
    Executa a metodologia de ajuste fino direto (Baseline).
    Treina um modelo YOLO pré-treinado estritamente nos
    conjuntos de dados de interiores de ônibus.
    """
    experiment_id: str = "yolo-baseline"
    weights: str = "yolo11m.pt"
    
    print(f"Iniciando o treinamento de base: {experiment_id}")
    
    data_spec: Dict[str, Any] = datasets.prepare(BUS_DATASETS, stage=DATASET_STAGE)
    model: YOLO = YOLO(weights)
    
    train.run(experiment_id, model, data_spec, TRAIN_CONFIG)

# Executar:
train_baseline()

Iniciando o treinamento de base: yolo-baseline


NameError: name 'datasets' is not defined

## 3.2 Metodologia 2: Ajuste Fino em Etapas (Robusto)

Uma abordagem de treinamento sequencial. A Etapa 1 utiliza o conjunto de dados `crowdhuman` para aprimorar a detecção em cenários com alta densidade de objetos. A Etapa 2 transfere os pesos aprendidos e os ajusta especificamente para o ambiente interno do ônibus.

In [ ]:
def train_robust_staged() -> None:
    f"""
    Executa a metodologia de ajuste fino robusto em dois estágios.
    O Estágio 1 treina em conjuntos de dados de multidões. O Estágio 2 ajusta os resultados do Estágio 1 em conjuntos de dados de ônibus.
    """
    # ---------------------------------------------------------
    # Etapa 1: Pré-treinamento no CrowdHuman
    # ---------------------------------------------------------
    stage1_id: str = "yolo-robust-stage1-crowd"
    stage1_weights: str = "yolo11m.pt"
    
    print(f"Starting Robust Stage 1: {stage1_id}")
    data_spec_stage1: Dict[str, Any] = datasets.prepare(CROWD_DATASETS, stage=DATASET_STAGE)
    model_stage1: YOLO = YOLO(stage1_weights)
    
    train.run(stage1_id, model_stage1, data_spec_stage1, TRAIN_CONFIG)
    
    # ---------------------------------------------------------
    # Transição: Localize os melhores pesos gerados pela Etapa 1.
    # ---------------------------------------------------------
    run_dir: Optional[Path]
    best_stage1_weights_path: Optional[str]
    run_dir, best_stage1_weights_path = eval_mod.find_run(stage1_id)
    
    if not best_stage1_weights_path:
        print("Erro: Não foi possível encontrar os pesos da Etapa 1. A Etapa 2 será abortada.")
        return
        
    # ---------------------------------------------------------
    # Etapa 2: Ajuste fino em conjuntos de dados de ônibus
    # ---------------------------------------------------------
    stage2_id: str = "yolo-robust-stage2-bus"
    
    print(f"Iniciando o Estágio Robusto 2: {stage2_id} usando pesos de {best_stage1_weights_path}")
    data_spec_stage2: Dict[str, Any] = datasets.prepare(BUS_DATASETS, stage=DATASET_STAGE)
    
    # Inicialize o modelo com os pesos adaptados ao domínio da Etapa 1.
    model_stage2: YOLO = YOLO(best_stage1_weights_path)
    train.run(stage2_id, model_stage2, data_spec_stage2, TRAIN_CONFIG)

# Executar:
train_robust_staged()

Starting Robust Stage 1: yolo-robust-stage1-crowd
train: 'run dir' -> /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/runs/yolo-robust-stage1-crowd_20260618_195910


wandb: run 'yolo-robust-stage1-crowd_20260618_195910' -> https://wandb.ai/IA901-2026S1-bus-passenger-count/bus-passenger-count/runs/dy1hd876
New https://pypi.org/project/ultralytics/8.4.71 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.50 🚀 Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA L40S, 45458MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/runs/yolo-robust-stage1-crowd_20260618_195910/_data_runtime.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=

[W618 19:59:12.982138021 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:12.991671057 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:12.008916676 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:12.011265797 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:12.015709881 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:12.066179048 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:12.074106458 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:12.086638661 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:12.088833002 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:12.091841164 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:5

YOLO11m summary: 232 layers, 20,053,779 parameters, 20,053,763 gradients, 68.2 GFLOPs



[W618 19:59:13.384730515 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:13.385191196 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:13.388144534 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:13.401819685 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:13.411266229 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:13.413227133 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:13.418150033 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:13.420727719 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:13.421409344 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:59:13.422926692 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 19:5

...

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1486.7±420.8 MB/s, size: 160.2 KB)
train: Scanning /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/data/processed/inside-bus-view/train/labels.cache... 513 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 513/513 143.4Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 271.8±127.8 MB/s, size: 30.9 KB)
val: Scanning /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/data/processed/inside-bus-view/valid/labels.cache... 109 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 109/109 3.3Mit/s 0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 106 weight(decay=0.0), 113 weight(decay=0.0005), 112 bias(decay=0.0)
Plotting labels to /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/runs/yolo-robust-stage2-bus_2

[W618 20:04:10.043490658 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 20:04:10.044387005 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 20:04:10.045133267 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 20:04:10.045414888 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 20:04:10.045598962 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 20:04:10.045812565 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 20:04:10.046034631 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 20:04:10.046231969 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 20:04:10.046448832 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 20:04:10.046638974 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W618 20:0

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.8it/s 0.5s
                   all        109       1045      0.338      0.417      0.305     0.0998
Speed: 0.1ms preprocess, 2.2ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/runs/yolo-robust-stage2-bus_20260618_200216/train
train: completado em 116.8s
train: 17 epochs conectado no wandb


final/train/duration_sec,▁
lr/pg0,▁▂▃▃▄▅▆▆▇▇████▇▇▇
lr/pg1,▁▂▃▃▄▅▆▆▇▇████▇▇▇
lr/pg2,▁▂▃▃▄▅▆▆▇▇████▇▇▇
metrics/mAP50(B),▅█▄▁▁▁▁▁▁▁▁▁▁▁▁▁▄
metrics/mAP50-95(B),▆█▅▁▁▁▁▁▁▁▁▁▁▁▁▁▄
metrics/precision(B),▇█▆▁▁▁▁▁▁▁▁▁▁▁▁▂▆
metrics/recall(B),▆██▂▁▁▁▁▃▁▂▁▁▁▂▂▇
time,▁▁▂▂▃▃▄▄▄▅▅▆▆▇▇██
train/box_loss,█▃▂▂▃▃▁▃▄▂▂▃▂▃▂▂▂
+6,...


wandb: execução(run) finalizada


## 3.3 Ajuste de Hiperparâmetros (Busca em Grade)

Exploração sistemática de hiperparâmetros para identificar a configuração ideal para o domínio interno do barramento.

Cada configuração é executada como um experimento separado e registrada em Pesos e Vieses para análise comparativa.

In [ ]:
def run_hyperparameter_tuning() -> None:
    f"""
    Iterates over a predefined grid of hyperparameters to find the optimal training configuration.
    Logs all variations to Weights & Biases for visualization.
    """
    print("Starting Hyperparameter Tuning...")
    
    base_weights: str = "yolo11m.pt"
    data_spec: Dict[str, Any] = datasets.prepare(BUS_DATASETS, stage=DATASET_STAGE)
    
    # Define the variations you want to test. Keep epochs low (e.g., 20) for faster iterations.
    tuning_grid: List[Dict[str, Any]] = [
        {"lr0": 0.01, "batch": 64, "optimizer": "auto", "mosaic": 1.0, "erasing": 0.4},
        {"lr0": 0.001, "batch": 64, "optimizer": "AdamW", "mosaic": 0.5, "erasing": 0.2},
        {"lr0": 0.01, "batch": 64, "optimizer": "SGD", "mosaic": 1.0, "erasing": 0.0},
        {"lr0": 0.005, "batch": 64, "optimizer": "auto", "mosaic": 0.0, "erasing": 0.4},
    ]
    
    for idx, params in enumerate(tuning_grid):
        # Create a dynamic experiment ID based on the parameters being tested
        lr_val: float = params["lr0"]
        opt_val: str = params["optimizer"]
        experiment_id: str = f"yolo-tune-lr{lr_val}-{opt_val}-v{idx}"
        
        print(f"\n--- Tuning Run {idx + 1}/{len(tuning_grid)}: {experiment_id} ---")
        
        # Merge the base config with the specific tuning parameters
        current_config: Dict[str, Any] = TRAIN_CONFIG.copy()
        current_config["epochs"] = 20 # Lower epochs for tuning
        current_config.update(params)
        
        # Initialize a fresh model for each run
        model: YOLO = YOLO(base_weights)
        
        # Execute the training run
        train.run(experiment_id, model, data_spec, current_config)
        
    print("Hyperparameter tuning complete. Check Weights & Biases dashboard for comparisons.")

# Execute Hyperparameter Tuning (Uncomment to run)
run_hyperparameter_tuning()